# Pipeline Bronze para Silver — CineData Analytics

Este notebook implementa o pipeline de conformação e enriquecimento de dados da camada **Bronze** para a camada **Silver**.

### Diretrizes de Engenharia e Regras Globais:
- **Validação Canônica de Esquemas**: Definição e aplicação de `StructType` com `StructField` tipados para cada tabela de destino.
- **Constantes de Colunas com Acesso Estático**: Utilização de `@dataclass(frozen=True)` acessadas diretamente na classe, eliminando instâncias intermediárias.
- **Tradução e Nomenclatura**: Nomes de colunas padronizados em português (`snake_case`).
- **Tipagem Segura**: Conversão explícita com proteção contra anomalias estruturais (*column shift* e resíduos espúrios).
- **Rastreabilidade**: Inclusão de `ingestion_datetime` com timestamp atualizado de gravação em todas as tabelas Silver.
- **Alta Coesão e Modularidade**: Funções auxiliares, esquemas e constantes declaradas localmente nas seções de cada tabela.

In [ ]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

from pyspark.sql import Column, DataFrame, SparkSession, Window
from pyspark.sql.functions import (
    coalesce,
    col,
    create_map,
    explode,
    expr,
    initcap,
    last,
    length,
    lit,
    regexp_extract,
    regexp_replace,
    row_number,
    split,
    to_date,
    trim,
    try_to_date,
    upper,
    when,
    year,
)
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import min as spark_min
from pyspark.sql.types import (
    DateType,
    DecimalType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

DATA_DIR = Path.cwd() / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"

SILVER_DIR.mkdir(exist_ok=True)

DEFAULT_SENTINEL_VALUES = [
    "UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "NONE", "N/A", "NULL", ""
]

spark = (
    SparkSession
    .builder
    .appName("Bronze_to_Silver")
    .getOrCreate()
)

def enforce_dataframe_schema(
    dataframe: DataFrame,
    expected_schema: StructType,
    strict_columns: bool = True
) -> DataFrame:
    """
    Valida a presença de todos os campos definidos no StructType, aplica conversão defensiva
    e reordena as colunas. Em modo estrito, impede colunas não declaradas no contrato.
    """
    actual_column_names = set(dataframe.columns)
    expected_column_names = [field.name for field in expected_schema.fields]
    missing_columns = set(expected_column_names) - actual_column_names

    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes no DataFrame: {missing_columns}")

    if strict_columns:
        unexpected_columns = actual_column_names - set(expected_column_names)
        if unexpected_columns:
            raise ValueError(f"Colunas imprevistas encontradas no DataFrame: {unexpected_columns}")

    ordered_column_expressions = [
        col(field.name).cast(field.dataType).alias(field.name)
        for field in expected_schema.fields
    ]
    return dataframe.select(ordered_column_expressions)

def write_dataframe(
    dataframe: DataFrame,
    target_path: Path,
    storage_format: str = "parquet",
    save_mode: str = "overwrite"
) -> None:
    """
    Persiste o DataFrame no formato especificado preservando o schema existente.
    """
    (
        dataframe.write
        .format(storage_format)
        .mode(save_mode)
        .save(str(target_path))
    )

def deduplicate_latest(
    dataframe: DataFrame,
    business_key_column: str,
    timestamp_column: str = "ingestion_datetime"
) -> DataFrame:
    """
    Garante a unicidade dos registros pela chave de negócio, mantendo exclusivamente
    a versão mais recente com base no timestamp de ingestão.
    """
    order_by_timestamp_descending = col(timestamp_column).desc()
    window_by_business_key = (
        Window
        .partitionBy(business_key_column)
        .orderBy(order_by_timestamp_descending)
    )
    return (
        dataframe
        .withColumn("_row_order_number", row_number().over(window_by_business_key))
        .filter(col("_row_order_number") == 1)
        .drop("_row_order_number")
    )

def try_cast(
    column: Column,
    target_data_type: str
) -> Column:
    """
    Conversão de tipo segura (safe casting) que retorna NULL para registros corrompidos ou deslocados.
    """
    return column.cast(target_data_type)


## 1. silver.tb_cotacao_dolar (Origem: bronze.tb_cotacao_dolar)

### Instruções e Regras de Negócio:
- **Série Temporal Contínua**: Como a API do Banco Central não possui cotações em finais de semana e feriados, estruture o histórico de forma a garantir uma série temporal contínua cobrindo todo o intervalo de datas.
- **Forward Fill**: Aplique uma técnica de preenchimento (*Forward Fill*) de modo que dias sem negociação recebam o valor da cotação do último dia útil disponível.
- **Precedência de Carga**: Esta tabela é processada primeiramente para disponibilizar a taxa cambial conformada para o cálculo financeiro em Reais (BRL).

In [ ]:
@dataclass(frozen=True)
class RawBacenCotacaoColumns:
    COTACAO_COMPRA: str = "cotacaoCompra"
    DATA_HORA_COTACAO: str = "dataHoraCotacao"
    INGESTION_DATETIME: str = "ingestion_datetime"

@dataclass(frozen=True)
class TbCotacaoDolarSilverColumns:
    DATA_COTACAO: str = "data_cotacao"
    COTACAO_COMPRA: str = "cotacao_compra"
    INGESTION_DATETIME: str = "ingestion_datetime"

EXCHANGE_RATE_PRECISION = "decimal(18,4)"

TbCotacaoDolarSilverSchema = StructType([
    StructField(TbCotacaoDolarSilverColumns.DATA_COTACAO, DateType(), nullable=False),
    StructField(TbCotacaoDolarSilverColumns.COTACAO_COMPRA, DecimalType(18, 4), nullable=True),
    StructField(TbCotacaoDolarSilverColumns.INGESTION_DATETIME, TimestampType(), nullable=True),
])

def transform_tb_cotacao_dolar(raw_cotacao_dataframe: DataFrame) -> DataFrame:
    """
    Estrutura uma série temporal contínua da cotação do dólar aplicando Forward Fill
    para dias sem negociação (finais de semana e feriados) e preservando o timestamp de ingestão.
    """
    col_input_quote = col(RawBacenCotacaoColumns.COTACAO_COMPRA)
    col_input_datetime = col(RawBacenCotacaoColumns.DATA_HORA_COTACAO)
    col_input_ingestion = col(RawBacenCotacaoColumns.INGESTION_DATETIME)

    expr_data_cotacao = to_date(col_input_datetime).alias(TbCotacaoDolarSilverColumns.DATA_COTACAO)
    expr_cotacao_bruta = col_input_quote.cast(EXCHANGE_RATE_PRECISION).alias("cotacao_compra_bruta")
    expr_ingestion = col_input_ingestion.alias(TbCotacaoDolarSilverColumns.INGESTION_DATETIME)

    # Garante unicidade diária selecionando a cotação e ingestão mais recentes do dia
    window_daily_quote = (
        Window
        .partitionBy(TbCotacaoDolarSilverColumns.DATA_COTACAO)
        .orderBy(col_input_datetime.desc(), col_input_ingestion.desc())
    )

    daily_quotes_dataframe = (
        raw_cotacao_dataframe
        .withColumn(TbCotacaoDolarSilverColumns.DATA_COTACAO, to_date(col_input_datetime))
        .withColumn("_row_order_number", row_number().over(window_daily_quote))
        .filter(col("_row_order_number") == 1)
        .select(
            col(TbCotacaoDolarSilverColumns.DATA_COTACAO),
            expr_cotacao_bruta,
            expr_ingestion
        )
    )
    
    # Cria calendário contínuo para cobrir fins de semana e feriados
    date_bounds = daily_quotes_dataframe.select(
        spark_min(TbCotacaoDolarSilverColumns.DATA_COTACAO),
        spark_max(TbCotacaoDolarSilverColumns.DATA_COTACAO)
    ).first()
    minimum_date, maximum_date = date_bounds[0], date_bounds[1]
    
    continuous_calendar_dataframe = spark.sql(
        f"SELECT explode(sequence(to_date('{minimum_date}'), to_date('{maximum_date}'), interval 1 day)) as {TbCotacaoDolarSilverColumns.DATA_COTACAO}"
    )
    
    # Forward Fill: dias sem negociação herdam a cotação e timestamp do último dia útil disponível
    forward_fill_window = Window.orderBy(TbCotacaoDolarSilverColumns.DATA_COTACAO).rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
    expr_filled_quote = last("cotacao_compra_bruta", ignorenulls=True).over(forward_fill_window)
    expr_filled_ingestion = last(TbCotacaoDolarSilverColumns.INGESTION_DATETIME, ignorenulls=True).over(forward_fill_window)

    transformed_dataframe = (
        continuous_calendar_dataframe
        .join(daily_quotes_dataframe, on=TbCotacaoDolarSilverColumns.DATA_COTACAO, how="left")
        .withColumn(TbCotacaoDolarSilverColumns.COTACAO_COMPRA, expr_filled_quote)
        .withColumn(TbCotacaoDolarSilverColumns.INGESTION_DATETIME, expr_filled_ingestion)
        .select(
            col(TbCotacaoDolarSilverColumns.DATA_COTACAO),
            col(TbCotacaoDolarSilverColumns.COTACAO_COMPRA),
            col(TbCotacaoDolarSilverColumns.INGESTION_DATETIME)
        )
    )
    return enforce_dataframe_schema(transformed_dataframe, TbCotacaoDolarSilverSchema)

dataframe_cotacao_dolar_bronze = spark.read.parquet(str(BRONZE_DIR / "bronze.tb_cotacao_dolar"))
dataframe_cotacao_dolar_silver = transform_tb_cotacao_dolar(dataframe_cotacao_dolar_bronze)

write_dataframe(
    dataframe=dataframe_cotacao_dolar_silver,
    target_path=SILVER_DIR / "silver.tb_cotacao_dolar"
)

dataframe_cotacao_dolar_silver.printSchema()
dataframe_cotacao_dolar_silver.show(10, truncate=False)


## 2. silver.tb_info_filmes (Origem: bronze.tb_movies_info)

### Instruções e Regras de Negócio:
- **Deduplicação**: A tabela deve conter unicidade por filme. Havendo registros duplicados na origem, mantenha exclusivamente a versão mais recente com base na data de ingestão.
- **Limpeza e Tradução do Status**: Normalizar a coluna de status (removendo ruídos, hífens sobressalentes e padronizando a caixa) antes da tradução dos termos para o português (`Released` → `Lançado`, etc.). Registros corrompidos ou não mapeáveis devem ser padronizados como `"Não Informado"`.
- **Tratamento de Data Multi-Formato**: Converter o campo de data de lançamento testando os diferentes padrões presentes na origem de forma robusta. Apenas valores onde a conversão for estritamente impossível devem ser tratados como ausentes (`NULL`).
- **Coluna Derivada**: Criar a coluna `ano_lancamento`, extraída de `data_lancamento`.

In [ ]:
@dataclass(frozen=True)
class RawMoviesInfoColumns:
    ID: str = "id"
    TITLE: str = "title"
    ORIGINAL_TITLE: str = "original_title"
    RELEASE_DATE: str = "release_date"
    RUNTIME: str = "runtime"
    ORIGINAL_LANGUAGE: str = "original_language"
    STATUS: str = "status"
    OVERVIEW: str = "overview"
    TAGLINE: str = "tagline"
    INGESTION_DATETIME: str = "ingestion_datetime"

@dataclass(frozen=True)
class TbInfoFilmesSilverColumns:
    MOVIE_ID: str = "id_filme"
    TITLE: str = "titulo"
    ORIGINAL_TITLE: str = "titulo_original"
    RELEASE_DATE: str = "data_lancamento"
    RELEASE_YEAR: str = "ano_lancamento"
    RUNTIME_MINUTES: str = "duracao_minutos"
    ORIGINAL_LANGUAGE: str = "idioma_original"
    STATUS: str = "status_filme"
    OVERVIEW: str = "sinopse"
    TAGLINE: str = "frase_divulgacao"
    INGESTION_DATETIME: str = "ingestion_datetime"

DEFAULT_STATUS_FALLBACK = "Não Informado"

STATUS_TRANSLATION_MAP = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado"
}

SUPPORTED_DATE_FORMATS = [
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd/MM/yyyy",
    "MM/dd/yyyy",
    "dd-MM-yyyy",
    "MM-dd-yyyy"
]

TbInfoFilmesSilverSchema = StructType([
    StructField(TbInfoFilmesSilverColumns.MOVIE_ID, IntegerType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.TITLE, StringType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.ORIGINAL_TITLE, StringType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.RELEASE_DATE, DateType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.RELEASE_YEAR, IntegerType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.RUNTIME_MINUTES, IntegerType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.ORIGINAL_LANGUAGE, StringType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.STATUS, StringType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.OVERVIEW, StringType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.TAGLINE, StringType(), nullable=True),
    StructField(TbInfoFilmesSilverColumns.INGESTION_DATETIME, TimestampType(), nullable=True),
])

def sanitize_string_column(column: Column) -> Column:
    """
    Higieniza colunas textuais, convertendo valores sentinela e strings vazias para NULL nativo.
    """
    uppercase_sentinels = [single_sentinel.upper() for single_sentinel in DEFAULT_SENTINEL_VALUES]
    trimmed_column = trim(column)
    uppercase_column = upper(trimmed_column)
    return when(uppercase_column.isin(uppercase_sentinels), lit(None)).otherwise(trimmed_column)

def clean_and_translate_status(column: Column) -> Column:
    """
    Normaliza a coluna de status (remoção de hífens, espaços múltiplos e caixa alta)
    antes da tradução dos termos para português, atribuindo valor default para registros corrompidos.
    """
    normalized_status_text = upper(trim(column))
    status_without_hyphens = regexp_replace(normalized_status_text, r"-", " ")
    sanitized_status_column = regexp_replace(status_without_hyphens, r"\s+", " ")
    
    spark_translation_map = create_map([
        lit(item)
        for key_value_pair in STATUS_TRANSLATION_MAP.items()
        for item in key_value_pair
    ])
    # Registros corrompidos, em branco ou não mapeáveis recebem o fallback 'Não Informado'
    return coalesce(spark_translation_map[sanitized_status_column], lit(DEFAULT_STATUS_FALLBACK))

def parse_multiformat_date(column: Column) -> Column:
    """
    Converte a coluna de data testando múltiplos padrões aceitos na base de origem.
    Valores estritamente impossíveis de converter resultam em NULL.
    """
    date_parsing_expressions = [
        try_to_date(column, single_date_format)
        for single_date_format in SUPPORTED_DATE_FORMATS
    ]
    return coalesce(*date_parsing_expressions)

def transform_tb_movies_info(raw_movies_dataframe: DataFrame) -> DataFrame:
    """
    Executa o pipeline de higienização, normalização e conformação da tabela de informações de filmes.
    """
    deduplicated_dataframe = deduplicate_latest(raw_movies_dataframe, business_key_column=RawMoviesInfoColumns.ID)

    col_input_id = col(RawMoviesInfoColumns.ID)
    col_input_title = col(RawMoviesInfoColumns.TITLE)
    col_input_original_title = col(RawMoviesInfoColumns.ORIGINAL_TITLE)
    col_input_release_date = col(RawMoviesInfoColumns.RELEASE_DATE)
    col_input_runtime = col(RawMoviesInfoColumns.RUNTIME)
    col_input_original_language = col(RawMoviesInfoColumns.ORIGINAL_LANGUAGE)
    col_input_status = col(RawMoviesInfoColumns.STATUS)
    col_input_overview = col(RawMoviesInfoColumns.OVERVIEW)
    col_input_tagline = col(RawMoviesInfoColumns.TAGLINE)
    col_input_ingestion = col(RawMoviesInfoColumns.INGESTION_DATETIME)

    expr_movie_id = try_cast(col_input_id, "integer").alias(TbInfoFilmesSilverColumns.MOVIE_ID)
    expr_title = sanitize_string_column(col_input_title).alias(TbInfoFilmesSilverColumns.TITLE)
    expr_original_title = sanitize_string_column(col_input_original_title).alias(TbInfoFilmesSilverColumns.ORIGINAL_TITLE)
    expr_release_date = parse_multiformat_date(col_input_release_date).alias(TbInfoFilmesSilverColumns.RELEASE_DATE)
    expr_release_year = year(parse_multiformat_date(col_input_release_date)).alias(TbInfoFilmesSilverColumns.RELEASE_YEAR)
    expr_runtime = try_cast(col_input_runtime, "integer").alias(TbInfoFilmesSilverColumns.RUNTIME_MINUTES)
    expr_original_language = sanitize_string_column(col_input_original_language).alias(TbInfoFilmesSilverColumns.ORIGINAL_LANGUAGE)
    expr_status = clean_and_translate_status(col_input_status).alias(TbInfoFilmesSilverColumns.STATUS)
    expr_overview = sanitize_string_column(col_input_overview).alias(TbInfoFilmesSilverColumns.OVERVIEW)
    expr_tagline = sanitize_string_column(col_input_tagline).alias(TbInfoFilmesSilverColumns.TAGLINE)
    expr_ingestion_datetime = col_input_ingestion.alias(TbInfoFilmesSilverColumns.INGESTION_DATETIME)

    transformed_dataframe = (
        deduplicated_dataframe
        .select(
            expr_movie_id,
            expr_title,
            expr_original_title,
            expr_release_date,
            expr_release_year,
            expr_runtime,
            expr_original_language,
            expr_status,
            expr_overview,
            expr_tagline,
            expr_ingestion_datetime
        )
    )
    return enforce_dataframe_schema(transformed_dataframe, TbInfoFilmesSilverSchema)

dataframe_movies_info_bronze = spark.read.parquet(str(BRONZE_DIR / "bronze.tb_movies_info"))
dataframe_movies_info_silver = transform_tb_movies_info(dataframe_movies_info_bronze)

write_dataframe(
    dataframe=dataframe_movies_info_silver,
    target_path=SILVER_DIR / "silver.tb_info_filmes"
)

dataframe_movies_info_silver.printSchema()
dataframe_movies_info_silver.show(5, truncate=False)


## 3. silver.tb_financeiro_filmes (Origem: bronze.tb_movies_financials)

### Instruções e Regras de Negócio:
- **Tratamento de Sentinelas**: Tratar valores textuais que representam ausência de dado (`Unknown`, `Não Informado`, etc.) como `NULL` antes da conversão de tipo.
- **Higienização Monetária**: Remover símbolos de moedas (`$`, `USD`), pontuações de milhar e converter notações abreviadas (`K`, `M`, `B`) para valores numéricos decimais (`DECIMAL(18,2)`).
- **Valores Inválidos**: Garantir que valores zerados ou negativos sejam tratados como dados ausentes (`NULL`).
- **Conversão Cambial Conforme a Camada Silver**: Obter a taxa de cotação mais recente a partir da tabela já conformada `silver.tb_cotacao_dolar` para converter os montantes para Reais (BRL).
- **Métricas Derivadas**: Derivar as colunas de Lucro (Dólar/Real) e Margem de Lucro Percentual, com tratamento seguro contra divisões por zero e propagação de nulos.

In [ ]:
@dataclass(frozen=True)
class RawMoviesFinancialsColumns:
    ID: str = "id"
    BUDGET: str = "budget"
    REVENUE: str = "revenue"
    INGESTION_DATETIME: str = "ingestion_datetime"

@dataclass(frozen=True)
class TbFinanceiroFilmesSilverColumns:
    MOVIE_ID: str = "id_filme"
    BUDGET_USD: str = "orcamento_usd"
    REVENUE_USD: str = "receita_usd"
    BUDGET_BRL: str = "orcamento_brl"
    REVENUE_BRL: str = "receita_brl"
    PROFIT_USD: str = "lucro_usd"
    PROFIT_BRL: str = "lucro_brl"
    PROFIT_MARGIN_PERCENT: str = "margem_lucro_percentual"
    INGESTION_DATETIME: str = "ingestion_datetime"

THOUSAND = 1_000
MILLION = 1_000_000
BILLION = 1_000_000_000
PERCENTAGE_FACTOR = 100
DECIMAL_PRECISION = "decimal(18,2)"

TbFinanceiroFilmesSilverSchema = StructType([
    StructField(TbFinanceiroFilmesSilverColumns.MOVIE_ID, IntegerType(), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.BUDGET_USD, DecimalType(18, 2), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.REVENUE_USD, DecimalType(18, 2), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.BUDGET_BRL, DecimalType(18, 2), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.REVENUE_BRL, DecimalType(18, 2), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.PROFIT_USD, DecimalType(18, 2), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.PROFIT_BRL, DecimalType(18, 2), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.PROFIT_MARGIN_PERCENT, DecimalType(18, 2), nullable=True),
    StructField(TbFinanceiroFilmesSilverColumns.INGESTION_DATETIME, TimestampType(), nullable=True),
])

def sanitize_currency_column(column: Column) -> Column:
    """
    Higieniza strings monetárias com símbolos ($, USD), valores sentinela
    e notações de escala (K, M, B), convertendo para DECIMAL(18, 2).
    Valores <= 0 ou corrompidos retornam NULL.
    """
    uppercase_sentinels = [single_sentinel.upper() for single_sentinel in DEFAULT_SENTINEL_VALUES]
    raw_column_text = upper(trim(column))
    
    # Converte textos que representam ausência de dado para NULL antes da extração de valores
    column_with_null_sentinels = when(raw_column_text.isin(uppercase_sentinels), lit(None)).otherwise(raw_column_text)
    
    # Remove caracteres monetários e pontuações de milhar
    column_without_currency_symbols = regexp_replace(column_with_null_sentinels, r"[$\s]|USD", "")
    column_without_thousand_separators = regexp_replace(column_without_currency_symbols, r",", "")
    
    extracted_thousands = regexp_extract(column_without_thousand_separators, r"^([0-9.]+)[Kk]$", 1)
    extracted_millions = regexp_extract(column_without_thousand_separators, r"^([0-9.]+)[Mm]$", 1)
    extracted_billions = regexp_extract(column_without_thousand_separators, r"^([0-9.]+)[Bb]$", 1)
    extracted_plain_number = regexp_extract(column_without_thousand_separators, r"^(-?[0-9.]+)$", 1)
    
    calculated_numeric_value = (
        when(extracted_thousands != "", extracted_thousands.cast("double") * THOUSAND)
        .when(extracted_millions != "", extracted_millions.cast("double") * MILLION)
        .when(extracted_billions != "", extracted_billions.cast("double") * BILLION)
        .when(extracted_plain_number != "", extracted_plain_number.cast("double"))
        .otherwise(lit(None))
    )
    
    # Conforme regra de negócio, orçamentos e receitas zerados ou negativos são tratados como NULL
    return when(calculated_numeric_value > 0, calculated_numeric_value.cast(DECIMAL_PRECISION)).otherwise(lit(None))

def calculate_safe_percentage(
    numerator_column: Column,
    denominator_column: Column,
    scale_factor: int = PERCENTAGE_FACTOR,
    target_precision: str = DECIMAL_PRECISION
) -> Column:
    """
    Calcula percentual de forma segura ((numerador / denominador) * scale_factor),
    protegendo contra divisões por zero e propagando NULL para valores ausentes.
    """
    valid_division_condition = (
        numerator_column.isNotNull()
        & denominator_column.isNotNull()
        & (denominator_column > 0)
    )
    percentage_calculation = ((numerator_column / denominator_column) * scale_factor).cast(target_precision)
    return when(valid_division_condition, percentage_calculation).otherwise(lit(None))

def transform_tb_movies_financials(
    raw_financials_dataframe: DataFrame,
    exchange_rate_dollar_to_brl: float
) -> DataFrame:
    """
    Aplica o pipeline completo de transformações e regras de negócio para a tabela financeira.
    """
    exchange_rate_expression = lit(exchange_rate_dollar_to_brl).cast(DECIMAL_PRECISION)
    deduplicated_dataframe = deduplicate_latest(raw_financials_dataframe, business_key_column=RawMoviesFinancialsColumns.ID)
    
    col_input_id = col(RawMoviesFinancialsColumns.ID)
    col_input_budget = col(RawMoviesFinancialsColumns.BUDGET)
    col_input_revenue = col(RawMoviesFinancialsColumns.REVENUE)
    col_input_ingestion = col(RawMoviesFinancialsColumns.INGESTION_DATETIME)

    sanitized_dataframe = (
        deduplicated_dataframe
        .withColumn(TbFinanceiroFilmesSilverColumns.MOVIE_ID, try_cast(col_input_id, "integer"))
        .withColumn(TbFinanceiroFilmesSilverColumns.BUDGET_USD, sanitize_currency_column(col_input_budget))
        .withColumn(TbFinanceiroFilmesSilverColumns.REVENUE_USD, sanitize_currency_column(col_input_revenue))
    )
    
    col_budget_usd = col(TbFinanceiroFilmesSilverColumns.BUDGET_USD)
    col_revenue_usd = col(TbFinanceiroFilmesSilverColumns.REVENUE_USD)
    
    converted_dataframe = (
        sanitized_dataframe
        .withColumn(TbFinanceiroFilmesSilverColumns.BUDGET_BRL, (col_budget_usd * exchange_rate_expression).cast(DECIMAL_PRECISION))
        .withColumn(TbFinanceiroFilmesSilverColumns.REVENUE_BRL, (col_revenue_usd * exchange_rate_expression).cast(DECIMAL_PRECISION))
    )
    
    col_budget_brl = col(TbFinanceiroFilmesSilverColumns.BUDGET_BRL)
    col_revenue_brl = col(TbFinanceiroFilmesSilverColumns.REVENUE_BRL)
    
    profit_dataframe = (
        converted_dataframe
        .withColumn(TbFinanceiroFilmesSilverColumns.PROFIT_USD, (col_revenue_usd - col_budget_usd).cast(DECIMAL_PRECISION))
        .withColumn(TbFinanceiroFilmesSilverColumns.PROFIT_BRL, (col_revenue_brl - col_budget_brl).cast(DECIMAL_PRECISION))
    )
    
    col_profit_usd = col(TbFinanceiroFilmesSilverColumns.PROFIT_USD)
    profit_margin_expr = calculate_safe_percentage(
        numerator_column=col_profit_usd,
        denominator_column=col_revenue_usd,
        target_precision=DECIMAL_PRECISION
    )
    
    expr_movie_id = col(TbFinanceiroFilmesSilverColumns.MOVIE_ID)
    expr_budget_usd = col_budget_usd
    expr_revenue_usd = col_revenue_usd
    expr_budget_brl = col_budget_brl
    expr_revenue_brl = col_revenue_brl
    expr_profit_usd = col_profit_usd
    expr_profit_brl = col(TbFinanceiroFilmesSilverColumns.PROFIT_BRL)
    expr_profit_margin = profit_margin_expr.alias(TbFinanceiroFilmesSilverColumns.PROFIT_MARGIN_PERCENT)
    expr_ingestion_datetime = col_input_ingestion.alias(TbFinanceiroFilmesSilverColumns.INGESTION_DATETIME)

    transformed_dataframe = (
        profit_dataframe
        .select(
            expr_movie_id,
            expr_budget_usd,
            expr_revenue_usd,
            expr_budget_brl,
            expr_revenue_brl,
            expr_profit_usd,
            expr_profit_brl,
            expr_profit_margin,
            expr_ingestion_datetime
        )
    )
    return enforce_dataframe_schema(transformed_dataframe, TbFinanceiroFilmesSilverSchema)

# Extração da taxa PTAX mais recente da tabela silver.tb_cotacao_dolar conformada
dataframe_cotacao_dolar_silver = spark.read.parquet(str(SILVER_DIR / "silver.tb_cotacao_dolar"))
latest_dollar_exchange_rate = (
    dataframe_cotacao_dolar_silver
    .orderBy(col(TbCotacaoDolarSilverColumns.DATA_COTACAO).desc())
    .select(TbCotacaoDolarSilverColumns.COTACAO_COMPRA)
    .first()[0]
)
print(f"Taxa de Cotação utilizada (PTAX Compra - Silver): R$ {latest_dollar_exchange_rate:.4f}")

dataframe_movies_financials_bronze = spark.read.parquet(str(BRONZE_DIR / "bronze.tb_movies_financials"))
dataframe_movies_financials_silver = transform_tb_movies_financials(
    raw_financials_dataframe=dataframe_movies_financials_bronze,
    exchange_rate_dollar_to_brl=float(latest_dollar_exchange_rate)
)

write_dataframe(
    dataframe=dataframe_movies_financials_silver,
    target_path=SILVER_DIR / "silver.tb_financeiro_filmes"
)

dataframe_movies_financials_silver.printSchema()
dataframe_movies_financials_silver.show(5, truncate=False)


## 4. silver.tb_metricas_engajamento (Origem: bronze.tb_movies_metrics)

### Instruções e Regras de Negócio:
- **Limpeza Numérica de Popularidade**: A coluna de popularidade apresenta inconsistências de pontuação e separadores decimais na origem. Limpe a formatação numérica da coluna antes da conversão de tipo.
- **Safe Casting contra Column Shift**: Devido ao deslocamento de colunas presente na base bruta, aplique uma conversão de tipagem segura, garantindo que textos ou caracteres incompatíveis sejam tratados como dados ausentes (`NULL`) sem interromper a execução do pipeline.
- **Validação de Limites de Negócio e Escala**:
  - Notas médias (TMDB e IMDb) fora do intervalo válido de 0 a 10 (incluindo valores multiplicados por erro de escala) devem ser desconsideradas e tratadas como `NULL`.
  - Contagens de votos ou índices de popularidade com valores negativos devem ser invalidados e tratados como `NULL`.

In [ ]:
@dataclass(frozen=True)
class RawMoviesMetricsColumns:
    ID: str = "id"
    POPULARITY: str = "popularity"
    VOTE_AVERAGE: str = "vote_average"
    VOTE_COUNT: str = "vote_count"
    AVERAGE_RATING: str = "averageRating"
    NUM_VOTES: str = "numVotes"
    INGESTION_DATETIME: str = "ingestion_datetime"

@dataclass(frozen=True)
class TbMetricasEngajamentoSilverColumns:
    MOVIE_ID: str = "id_filme"
    POPULARITY: str = "popularidade"
    VOTE_AVERAGE_TMDB: str = "nota_media_tmdb"
    VOTE_COUNT_TMDB: str = "qtd_votos_tmdb"
    VOTE_AVERAGE_IMDB: str = "nota_media_imdb"
    VOTE_COUNT_IMDB: str = "qtd_votos_imdb"
    INGESTION_DATETIME: str = "ingestion_datetime"

MINIMUM_RATING_VALUE = 0.0
MAXIMUM_RATING_VALUE = 10.0
MINIMUM_COUNT_VALUE = 0

TbMetricasEngajamentoSilverSchema = StructType([
    StructField(TbMetricasEngajamentoSilverColumns.MOVIE_ID, IntegerType(), nullable=True),
    StructField(TbMetricasEngajamentoSilverColumns.POPULARITY, DoubleType(), nullable=True),
    StructField(TbMetricasEngajamentoSilverColumns.VOTE_AVERAGE_TMDB, DoubleType(), nullable=True),
    StructField(TbMetricasEngajamentoSilverColumns.VOTE_COUNT_TMDB, IntegerType(), nullable=True),
    StructField(TbMetricasEngajamentoSilverColumns.VOTE_AVERAGE_IMDB, DoubleType(), nullable=True),
    StructField(TbMetricasEngajamentoSilverColumns.VOTE_COUNT_IMDB, IntegerType(), nullable=True),
    StructField(TbMetricasEngajamentoSilverColumns.INGESTION_DATETIME, TimestampType(), nullable=True),
])

def sanitize_numeric_metric(
    column: Column,
    target_data_type: str,
    minimum_value: float | None = None,
    maximum_value: float | None = None
) -> Column:
    """
    Higieniza métricas numéricas com substituição de vírgula por ponto,
    conversão segura de tipo (try_cast) e validação de limites de negócio.
    Valores fora dos limites ou não numéricos retornam NULL.
    """
    sanitized_expression = expr(f"try_cast(replace(trim({column._jc.toString()}), ',', '.') as {target_data_type})")
    
    is_valid_metric = sanitized_expression.isNotNull()
    if minimum_value is not None:
        is_valid_metric = is_valid_metric & (sanitized_expression >= lit(minimum_value))
    if maximum_value is not None:
        is_valid_metric = is_valid_metric & (sanitized_expression <= lit(maximum_value))
        
    return when(is_valid_metric, sanitized_expression).otherwise(lit(None))

def transform_tb_movies_metrics(raw_metrics_dataframe: DataFrame) -> DataFrame:
    """
    Higieniza, conforma e valida as métricas de popularidade e avaliação dos filmes.
    """
    deduplicated_metrics_dataframe = deduplicate_latest(raw_metrics_dataframe, business_key_column=RawMoviesMetricsColumns.ID)
    
    col_input_id = col(RawMoviesMetricsColumns.ID)
    col_input_popularity = col(RawMoviesMetricsColumns.POPULARITY)
    col_input_vote_avg_tmdb = col(RawMoviesMetricsColumns.VOTE_AVERAGE)
    col_input_vote_cnt_tmdb = col(RawMoviesMetricsColumns.VOTE_COUNT)
    col_input_vote_avg_imdb = col(RawMoviesMetricsColumns.AVERAGE_RATING)
    col_input_vote_cnt_imdb = col(RawMoviesMetricsColumns.NUM_VOTES)
    col_input_ingestion = col(RawMoviesMetricsColumns.INGESTION_DATETIME)

    expr_movie_id = try_cast(col_input_id, "integer").alias(TbMetricasEngajamentoSilverColumns.MOVIE_ID)
    expr_popularity = (
        when(
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.POPULARITY}), ',', '.') as double)") >= 0.0,
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.POPULARITY}), ',', '.') as double)")
        ).otherwise(lit(None))
        .alias(TbMetricasEngajamentoSilverColumns.POPULARITY)
    )
    expr_vote_avg_tmdb = (
        when(
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.VOTE_AVERAGE}), ',', '.') as double)").between(MINIMUM_RATING_VALUE, MAXIMUM_RATING_VALUE),
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.VOTE_AVERAGE}), ',', '.') as double)")
        ).otherwise(lit(None))
        .alias(TbMetricasEngajamentoSilverColumns.VOTE_AVERAGE_TMDB)
    )
    expr_vote_cnt_tmdb = (
        when(
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.VOTE_COUNT}), ',', '.') as integer)") >= MINIMUM_COUNT_VALUE,
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.VOTE_COUNT}), ',', '.') as integer)")
        ).otherwise(lit(None))
        .alias(TbMetricasEngajamentoSilverColumns.VOTE_COUNT_TMDB)
    )
    expr_vote_avg_imdb = (
        when(
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.AVERAGE_RATING}), ',', '.') as double)").between(MINIMUM_RATING_VALUE, MAXIMUM_RATING_VALUE),
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.AVERAGE_RATING}), ',', '.') as double)")
        ).otherwise(lit(None))
        .alias(TbMetricasEngajamentoSilverColumns.VOTE_AVERAGE_IMDB)
    )
    expr_vote_cnt_imdb = (
        when(
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.NUM_VOTES}), ',', '.') as integer)") >= MINIMUM_COUNT_VALUE,
            expr(f"try_cast(replace(trim({RawMoviesMetricsColumns.NUM_VOTES}), ',', '.') as integer)")
        ).otherwise(lit(None))
        .alias(TbMetricasEngajamentoSilverColumns.VOTE_COUNT_IMDB)
    )
    expr_ingestion_datetime = col_input_ingestion.alias(TbMetricasEngajamentoSilverColumns.INGESTION_DATETIME)

    transformed_dataframe = deduplicated_metrics_dataframe.select(
        expr_movie_id,
        expr_popularity,
        expr_vote_avg_tmdb,
        expr_vote_cnt_tmdb,
        expr_vote_avg_imdb,
        expr_vote_cnt_imdb,
        expr_ingestion_datetime
    )
    return enforce_dataframe_schema(transformed_dataframe, TbMetricasEngajamentoSilverSchema)

dataframe_movies_metrics_bronze = spark.read.parquet(str(BRONZE_DIR / "bronze.tb_movies_metrics"))
dataframe_movies_metrics_silver = transform_tb_movies_metrics(dataframe_movies_metrics_bronze)

write_dataframe(
    dataframe=dataframe_movies_metrics_silver,
    target_path=SILVER_DIR / "silver.tb_metricas_engajamento"
)

dataframe_movies_metrics_silver.printSchema()
dataframe_movies_metrics_silver.show(5, truncate=False)


## 5. silver.tb_avaliacoes_usuarios (Origem: bronze.tb_movies_reviews)

### Instruções e Regras de Negócio:
- **Deduplicação Integral**: Remova registros integralmente duplicados (onde a combinação de filme, usuário, nota e comentário seja idêntica), garantindo a unicidade das avaliações.
- **Validação de Nota**: Garanta que a nota atribuída pelo usuário respeite a regra de negócio da escala permitida (0 a 10). Qualquer valor fora dessa faixa deve ser descartado e convertido para `NULL`.
- **Preenchimento de Comentários**: Identifique comentários não preenchidos ou compostos apenas por espaços em branco e preencha-os de forma explícita com o texto padronizado `"Sem comentário"`.

In [ ]:
@dataclass(frozen=True)
class RawMoviesReviewsColumns:
    ID: str = "id"
    NOME: str = "nome"
    NOTA: str = "nota"
    COMENTARIO: str = "comentario"
    INGESTION_DATETIME: str = "ingestion_datetime"

@dataclass(frozen=True)
class TbAvaliacoesUsuariosSilverColumns:
    MOVIE_ID: str = "id_filme"
    USER_NAME: str = "nome_usuario"
    USER_RATING: str = "nota_usuario"
    USER_COMMENT: str = "comentario_usuario"
    INGESTION_DATETIME: str = "ingestion_datetime"

MINIMUM_USER_RATING = 0.0
MAXIMUM_USER_RATING = 10.0
DEFAULT_COMMENT_FALLBACK = "Sem comentário"

TbAvaliacoesUsuariosSilverSchema = StructType([
    StructField(TbAvaliacoesUsuariosSilverColumns.MOVIE_ID, IntegerType(), nullable=True),
    StructField(TbAvaliacoesUsuariosSilverColumns.USER_NAME, StringType(), nullable=True),
    StructField(TbAvaliacoesUsuariosSilverColumns.USER_RATING, DoubleType(), nullable=True),
    StructField(TbAvaliacoesUsuariosSilverColumns.USER_COMMENT, StringType(), nullable=True),
    StructField(TbAvaliacoesUsuariosSilverColumns.INGESTION_DATETIME, TimestampType(), nullable=True),
])

def transform_tb_movies_reviews(raw_reviews_dataframe: DataFrame) -> DataFrame:
    """
    Executa a deduplicação integral, validação da escala de notas de usuários e padronização de comentários.
    """
    deduplicated_reviews_dataframe = raw_reviews_dataframe.dropDuplicates([
        RawMoviesReviewsColumns.ID,
        RawMoviesReviewsColumns.NOME,
        RawMoviesReviewsColumns.NOTA,
        RawMoviesReviewsColumns.COMENTARIO
    ])
    
    col_input_id = col(RawMoviesReviewsColumns.ID)
    col_input_name = col(RawMoviesReviewsColumns.NOME)
    col_input_rating = col(RawMoviesReviewsColumns.NOTA)
    col_input_comment = trim(col(RawMoviesReviewsColumns.COMENTARIO))
    col_input_ingestion = col(RawMoviesReviewsColumns.INGESTION_DATETIME)

    expr_movie_id = try_cast(col_input_id, "integer").alias(TbAvaliacoesUsuariosSilverColumns.MOVIE_ID)
    expr_user_name = trim(col_input_name).alias(TbAvaliacoesUsuariosSilverColumns.USER_NAME)
    
    # Notas fora da escala permitida de 0 a 10 são convertidas para NULL
    valid_rating_condition = col_input_rating.between(MINIMUM_USER_RATING, MAXIMUM_USER_RATING)
    expr_user_rating = when(valid_rating_condition, col_input_rating).otherwise(lit(None)).alias(TbAvaliacoesUsuariosSilverColumns.USER_RATING)
    
    # Comentários em branco ou ausentes recebem o texto padronizado 'Sem comentário'
    valid_comment_condition = col_input_comment.isNotNull() & (col_input_comment != "")
    expr_user_comment = when(valid_comment_condition, col_input_comment).otherwise(lit(DEFAULT_COMMENT_FALLBACK)).alias(TbAvaliacoesUsuariosSilverColumns.USER_COMMENT)
    
    expr_ingestion_datetime = col_input_ingestion.alias(TbAvaliacoesUsuariosSilverColumns.INGESTION_DATETIME)

    transformed_dataframe = deduplicated_reviews_dataframe.select(
        expr_movie_id,
        expr_user_name,
        expr_user_rating,
        expr_user_comment,
        expr_ingestion_datetime
    )
    return enforce_dataframe_schema(transformed_dataframe, TbAvaliacoesUsuariosSilverSchema)

dataframe_movies_reviews_bronze = spark.read.parquet(str(BRONZE_DIR / "bronze.tb_movies_reviews"))
dataframe_movies_reviews_silver = transform_tb_movies_reviews(dataframe_movies_reviews_bronze)

write_dataframe(
    dataframe=dataframe_movies_reviews_silver,
    target_path=SILVER_DIR / "silver.tb_avaliacoes_usuarios"
)

dataframe_movies_reviews_silver.printSchema()
dataframe_movies_reviews_silver.show(5, truncate=False)


## 6. silver.tb_generos (Origem: bronze.tb_credits_and_tags, coluna `genres`)

### Instruções e Regras de Negócio:
- **Explosão Atômica**: Explodir (`split` + `explode`) a coluna `genres`, tratando a inconsistência de separadores (vírgula vs. ponto e vírgula vs. pipes) antes do split para que cada registro represente um único gênero por filme.
- **Filtragem de Column Shift**: Devido a falhas na estrutura de origem, remova resíduos em branco, textos descritivos e valores numéricos deslocados que não pertençam ao domínio canônico de gêneros cinematográficos.

In [ ]:
@dataclass(frozen=True)
class RawCreditsAndTagsColumns:
    ID: str = "id"
    GENRES: str = "genres"
    CAST: str = "cast"
    DIRECTORS: str = "directors"
    WRITERS: str = "writers"
    PRODUCTION_COMPANIES: str = "production_companies"
    INGESTION_DATETIME: str = "ingestion_datetime"

@dataclass(frozen=True)
class TbGenerosSilverColumns:
    MOVIE_ID: str = "id_filme"
    GENRE_NAME: str = "nome_genero"
    INGESTION_DATETIME: str = "ingestion_datetime"

KNOWN_GENRES_LIST = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music",
    "Mystery", "Romance", "Science Fiction", "TV Movie", "Thriller",
    "War", "Western"
]

TbGenerosSilverSchema = StructType([
    StructField(TbGenerosSilverColumns.MOVIE_ID, IntegerType(), nullable=True),
    StructField(TbGenerosSilverColumns.GENRE_NAME, StringType(), nullable=True),
    StructField(TbGenerosSilverColumns.INGESTION_DATETIME, TimestampType(), nullable=True),
])

def transform_tb_generos(raw_credits_dataframe: DataFrame) -> DataFrame:
    """
    Normaliza e explode os gêneros cinematográficos em grão atômico por filme,
    eliminando anomalias de formatação e fragmentos deslocados.
    """
    deduplicated_dataframe = deduplicate_latest(raw_credits_dataframe, business_key_column=RawCreditsAndTagsColumns.ID)
    
    col_input_id = col(RawCreditsAndTagsColumns.ID)
    col_input_genres = col(RawCreditsAndTagsColumns.GENRES)
    col_input_ingestion = col(RawCreditsAndTagsColumns.INGESTION_DATETIME)

    # Normaliza separadores múltiplos (vírgula, ponto e vírgula, pipes) antes do desmembramento
    normalized_genres_text = regexp_replace(col_input_genres, r"[,;|]+", ",")
    genres_array_column = split(normalized_genres_text, ",")
    
    col_raw_genre = col("_raw_genre")
    sanitized_genre_name = trim(regexp_replace(col_raw_genre, r"""["'\[\]{}]""", ""))

    expr_movie_id = try_cast(col_input_id, "integer").alias(TbGenerosSilverColumns.MOVIE_ID)
    expr_genre_name = col(TbGenerosSilverColumns.GENRE_NAME)
    expr_ingestion_datetime = col_input_ingestion.alias(TbGenerosSilverColumns.INGESTION_DATETIME)

    transformed_dataframe = (
        deduplicated_dataframe
        .filter(col_input_genres.isNotNull())
        # Desmembra os múltiplos gêneros para que cada registro represente um único gênero por filme
        .withColumn("_raw_genre", explode(genres_array_column))
        .withColumn(TbGenerosSilverColumns.GENRE_NAME, sanitized_genre_name)
        # Filtra column shift e resíduos numéricos/textuais não pertencentes ao domínio canônico
        .filter(col(TbGenerosSilverColumns.GENRE_NAME).isin(KNOWN_GENRES_LIST))
        .select(
            expr_movie_id,
            expr_genre_name,
            expr_ingestion_datetime
        )
        .dropDuplicates([TbGenerosSilverColumns.MOVIE_ID, TbGenerosSilverColumns.GENRE_NAME])
    )
    return enforce_dataframe_schema(transformed_dataframe, TbGenerosSilverSchema)

dataframe_credits_bronze = spark.read.parquet(str(BRONZE_DIR / "bronze.tb_credits_and_tags"))
dataframe_generos_silver = transform_tb_generos(dataframe_credits_bronze)

write_dataframe(
    dataframe=dataframe_generos_silver,
    target_path=SILVER_DIR / "silver.tb_generos"
)

dataframe_generos_silver.printSchema()
dataframe_generos_silver.show(10, truncate=False)


## 7. silver.tb_pessoas_empresas (Origem: bronze.tb_credits_and_tags)

### Instruções e Regras de Negócio:
- **Dimensão Unificada**: Consolidar quatro tipos de entidade em uma única tabela:
  - `cast` $\rightarrow$ `'Ator'`
  - `directors` $\rightarrow$ `'Diretor'`
  - `writers` $\rightarrow$ `'Roteirista'`
  - `production_companies` $\rightarrow$ `'Produtora'`
- **Padronização e Deduplicação**: Padronize a formatação de capitalização de texto (`initcap`), consolide as entidades categorizadas pelo tipo de atuação e elimine registros duplicados e resíduos numéricos/URLs de *column shift*.

In [ ]:
@dataclass(frozen=True)
class TbPessoasEmpresasSilverColumns:
    MOVIE_ID: str = "id_filme"
    ENTITY_NAME: str = "nome_entidade"
    ENTITY_TYPE: str = "tipo_entidade"
    INGESTION_DATETIME: str = "ingestion_datetime"

ENTITY_TYPE_MAPPINGS = [
    (RawCreditsAndTagsColumns.CAST, "Ator"),
    (RawCreditsAndTagsColumns.DIRECTORS, "Diretor"),
    (RawCreditsAndTagsColumns.WRITERS, "Roteirista"),
    (RawCreditsAndTagsColumns.PRODUCTION_COMPANIES, "Produtora")
]

TbPessoasEmpresasSilverSchema = StructType([
    StructField(TbPessoasEmpresasSilverColumns.MOVIE_ID, IntegerType(), nullable=True),
    StructField(TbPessoasEmpresasSilverColumns.ENTITY_NAME, StringType(), nullable=True),
    StructField(TbPessoasEmpresasSilverColumns.ENTITY_TYPE, StringType(), nullable=True),
    StructField(TbPessoasEmpresasSilverColumns.INGESTION_DATETIME, TimestampType(), nullable=True),
])

def extract_and_clean_entities(
    dataframe: DataFrame,
    source_column_name: str,
    entity_type_label: str
) -> DataFrame:
    """
    Desmembra múltiplos participantes/empresas em linhas atômicas,
    padroniza a capitalização do texto (initcap), atribui o tipo de atuação
    e remove resíduos espúrios de column shift (URLs, imagens, números).
    """
    col_input_id = col(RawCreditsAndTagsColumns.ID)
    col_source = col(source_column_name)
    col_input_ingestion = col(RawCreditsAndTagsColumns.INGESTION_DATETIME)

    normalized_source_text = regexp_replace(col_source, r"[,;|]+", ",")
    entities_array = split(normalized_source_text, ",")
    
    col_raw_entity = col("_raw_entity")
    sanitized_entity_name = initcap(trim(regexp_replace(col_raw_entity, r"""["'\[\]{}]""", "")))

    expr_movie_id = try_cast(col_input_id, "integer").alias(TbPessoasEmpresasSilverColumns.MOVIE_ID)
    expr_entity_name = col(TbPessoasEmpresasSilverColumns.ENTITY_NAME)
    expr_entity_type = col(TbPessoasEmpresasSilverColumns.ENTITY_TYPE)
    expr_ingestion_datetime = col_input_ingestion.alias(TbPessoasEmpresasSilverColumns.INGESTION_DATETIME)

    return (
        dataframe
        .filter(col_source.isNotNull())
        .withColumn("_raw_entity", explode(entities_array))
        .withColumn(TbPessoasEmpresasSilverColumns.ENTITY_NAME, sanitized_entity_name)
        .withColumn(TbPessoasEmpresasSilverColumns.ENTITY_TYPE, lit(entity_type_label))
        # Elimina anomalias de column shift (URLs, imagens, números isolados e sentinelas)
        .filter(
            col(TbPessoasEmpresasSilverColumns.ENTITY_NAME).isNotNull()
            & (col(TbPessoasEmpresasSilverColumns.ENTITY_NAME) != "")
            & (~upper(col(TbPessoasEmpresasSilverColumns.ENTITY_NAME)).isin(DEFAULT_SENTINEL_VALUES))
            & (~col(TbPessoasEmpresasSilverColumns.ENTITY_NAME).rlike("^[0-9. -]+$"))
            & (~col(TbPessoasEmpresasSilverColumns.ENTITY_NAME).rlike(r"(?i)\.(jpg|png|jpeg|webp)"))
            & (~col(TbPessoasEmpresasSilverColumns.ENTITY_NAME).rlike(r"(?i)^(http|https|www\.)"))
            & (length(col(TbPessoasEmpresasSilverColumns.ENTITY_NAME)) > 1)
        )
        .select(
            expr_movie_id,
            expr_entity_name,
            expr_entity_type,
            expr_ingestion_datetime
        )
    )

def transform_tb_pessoas_empresas(raw_credits_dataframe: DataFrame) -> DataFrame:
    """
    Consolida atores, diretores, roteiristas e produtoras em uma única dimensão
    padronizada, categorizada e deduplicada.
    """
    deduplicated_credits_dataframe = deduplicate_latest(raw_credits_dataframe, business_key_column=RawCreditsAndTagsColumns.ID)
    
    extracted_entity_dataframes = [
        extract_and_clean_entities(
            dataframe=deduplicated_credits_dataframe,
            source_column_name=source_column,
            entity_type_label=target_label
        )
        for source_column, target_label in ENTITY_TYPE_MAPPINGS
    ]
    
    # Consolida as diferentes categorias de atuação em uma dimensão unificada
    unified_entities_dataframe = extracted_entity_dataframes[0]
    for additional_entity_dataframe in extracted_entity_dataframes[1:]:
        unified_entities_dataframe = unified_entities_dataframe.unionByName(additional_entity_dataframe)
        
    transformed_dataframe = unified_entities_dataframe.dropDuplicates([
        TbPessoasEmpresasSilverColumns.MOVIE_ID,
        TbPessoasEmpresasSilverColumns.ENTITY_NAME,
        TbPessoasEmpresasSilverColumns.ENTITY_TYPE
    ])
    return enforce_dataframe_schema(transformed_dataframe, TbPessoasEmpresasSilverSchema)

dataframe_pessoas_empresas_silver = transform_tb_pessoas_empresas(dataframe_credits_bronze)

write_dataframe(
    dataframe=dataframe_pessoas_empresas_silver,
    target_path=SILVER_DIR / "silver.tb_pessoas_empresas"
)

dataframe_pessoas_empresas_silver.printSchema()
dataframe_pessoas_empresas_silver.show(10, truncate=False)
